In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

files = glob.glob("stck_data/*.csv")

list_df = []
for file in files:
    df_temp = pd.read_csv(file)
    symbol = os.path.basename(file).split('.')[0]
    df_temp['Symbol'] = symbol
    list_df.append(df_temp)

df_all = pd.concat(list_df, ignore_index=True)

In [ ]:
df_all.head()

In [ ]:
df_all.isna().sum()

In [ ]:
numeric_cols = ['Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']

df_all[numeric_cols] = df_all[numeric_cols].ffill().bfill()

for col in numeric_cols:
    df_all[col] = df_all[col].str.replace(',', '', regex=True)

df_all['Vol.'] = (
    df_all['Vol.']
    .replace({'K': '*1e3', 'M': '*1e6'}, regex=True)
    .map(pd.eval)
    .astype(float)
)

In [ ]:
df_all.head()

In [ ]:
df_all['Change %'] = df_all['Change %'].str.replace('%', '', regex=True).astype(float)

for col in ['Price', 'Open', 'High', 'Low']:
    df_all[col] = df_all[col].astype(float)

df_all['Date'] = pd.to_datetime(df_all['Date'])

In [ ]:
len(df_all)

In [ ]:
df = df_all.sort_values(by=['Symbol', 'Date'])

In [ ]:
df['return_day'] = df.groupby('Symbol')['Price'].pct_change()
df['return_week'] = df.groupby('Symbol')['Price'].pct_change(periods=5)
df['return_month'] = df.groupby('Symbol')['Price'].pct_change(periods=22)

In [ ]:
df.head(23)

In [ ]:
df['volatility_day'] = df.groupby('Symbol')['return_day'].rolling(window=5).std().reset_index(level=0, drop=True)
df["volatility_week"] = df.groupby("Symbol")["return_day"].rolling(window=21).std().reset_index(level=0, drop=True)
df["volatility_month"] = df.groupby("Symbol")["return_day"].rolling(window=63).std().reset_index(level=0, drop=True)

In [ ]:
df['liquidity_day'] = df.groupby("Symbol")['Vol.'].rolling(window=5).mean().reset_index(level=0, drop=True)
df['liquidity_week'] = df.groupby("Symbol")['Vol.'].rolling(window=21).mean().reset_index(level=0, drop=True)   
df['liquidity_month'] = df.groupby("Symbol")['Vol.'].rolling(window=63).mean().reset_index(level=0, drop=True)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# First, calculate z-scores if you haven't already
def add_z_score(group):
    mean = group['Price'].mean()
    std = group['Price'].std()
    group['z_score'] = (group['Price'] - mean) / std
    return group

df.reset_index(inplace=True) 

df = df.groupby('Symbol', group_keys=False).apply(add_z_score)

In [ ]:
symbols = df['Symbol'].unique()

for symbol in symbols:
    df_symbol = df[df['Symbol'] == symbol]

    plt.figure(figsize=(14, 6))
    
    # Plot Price clearly
    plt.plot(df_symbol['Date'], df_symbol['Price'], color='blue', label='Price')

    # Identify and plot Outliers (z_score > ±3)
    outliers = df_symbol[np.abs(df_symbol['z_score']) > 3]
    plt.scatter(outliers['Date'], outliers['Price'], color='red', label='Outliers')

    plt.title(f'Price and Outliers for Symbol: {symbol}')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# # remove outliers
# df = df[df['z_score'].abs() < 3].copy()
# df.drop(columns=['z_score'], inplace=True)
# df.set_index(['Symbol', 'Date'], inplace=True)

In [ ]:
len(df)

In [ ]:
sma_windows = [20, 50, 100]
for window in sma_windows:
    df[f'SMA_{window}'] = df.groupby('Symbol')['Price'].transform(lambda x: x.rolling(window, min_periods=1).mean())
    
def weighted_moving_average(prices, window):
    weights = np.arange(1, window + 1)
    return prices.rolling(window).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

wma_windows = [20, 50, 100]
for window in wma_windows:
    df[f'WMA_{window}'] = df.groupby('Symbol')['Price'].transform(lambda x: weighted_moving_average(x, window))

In [ ]:
df['EMA12'] = df.groupby('Symbol')['Price'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
df['EMA26'] = df.groupby('Symbol')['Price'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
df['MACD'] = df['EMA12'] - df['EMA26']
df['Signal_Line'] = df.groupby('Symbol')['MACD'].transform(lambda x: x.ewm(span=9, adjust=False).mean())

In [ ]:
def RSI(series, period=14):
    delta = series.diff(1)
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    
    # Use exponential moving average for smoother results
    avg_gain = gain.ewm(span=period, adjust=False).mean()
    avg_loss = loss.ewm(span=period, adjust=False).mean()
    
    RS = avg_gain / avg_loss
    return 100 - (100 / (1 + RS))

df['RSI_14'] = df.groupby('Symbol')['Price'].transform(lambda x: RSI(x, 14))

In [ ]:
def stochastic_RSI(rsi_series, period=14):
    min_rsi = rsi_series.rolling(window=period).min()
    max_rsi = rsi_series.rolling(window=period).max()
    return (rsi_series - min_rsi) / (max_rsi - min_rsi)

df['StochRSI_14'] = df.groupby('Symbol')['RSI_14'].transform(lambda x: stochastic_RSI(x, 14))

In [ ]:
# --- Bollinger Bands (20 days, 2 std deviations) ---
df['BB_upper'] = df['SMA_20'] + 2 * df.groupby('Symbol')['Price'].transform(lambda x: x.rolling(20).std())
df['BB_lower'] = df['SMA_20'] - 2 * df.groupby('Symbol')['Price'].transform(lambda x: x.rolling(20).std())

In [ ]:
df.head(5)

In [ ]:
df.isna().sum() 

In [ ]:
# forward backward filling (could also think of linear interpolation)
indicator_cols = ['WMA_20', 'WMA_50', 'WMA_100', 'RSI_14', 'BB_upper', 'BB_lower', 'RSI_14', 'StochRSI_14', 'return_day', 'return_week', 'return_month', 'volatility_day', 'volatility_week', 'volatility_month', 'liquidity_day', 'liquidity_week', 'liquidity_month']

for col in indicator_cols:
    df[col] = df.groupby('Symbol')[col].transform(lambda x: x.ffill().bfill())

In [ ]:
df.head()

In [ ]:
df.drop(columns=['index'], inplace=True, errors='ignore')

In [ ]:
len(df)

In [ ]:
df.head()

In [ ]:
df.to_csv('processed_stock_data.csv', index=False)

# Micro/Marco Indicators

In [ ]:
def clean_macro_csv(file_path):
    df = pd.read_csv(file_path, skiprows=4)
    df.dropna(axis=1, how='all', inplace=True)

    df_long = df.melt(
        id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
        var_name="Year",
        value_name="Value"
    )
    
    df_long["Year"] = pd.to_numeric(df_long["Year"], errors="coerce").astype('Int64')
    df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")
    
    df_long = df_long.reset_index(drop=True)
    
    return df_long

gdp_file = "stck_data/Chỉ số vĩ mô/GDP/GDP.csv"
gdp_growth_file = 'stck_data/Chỉ số vĩ mô/GDP_growth/GDP_growth.csv'
gdp_per_capita_file = 'stck_data/Chỉ số vĩ mô/GDP_per_capita/GDP_per_capita.csv'
inflation_file = 'stck_data/Chỉ số vĩ mô/Inflation/Inflation.csv'
unemployment_file = 'stck_data/Chỉ số vĩ mô/Unemployment/Unemployment.csv'

gdp_df = clean_macro_csv(gdp_file)
gdp_growth_df = clean_macro_csv(gdp_growth_file)
gdp_per_capita_df = clean_macro_csv(gdp_per_capita_file)
inflation_df = clean_macro_csv(inflation_file)
unemployment_df = clean_macro_csv(unemployment_file)

target_country = "Viet Nam"

vn_df = gdp_df[gdp_df["Country Name"].str.strip().str.lower() == target_country.lower()]    
vn_gdp_growth = gdp_growth_df[gdp_growth_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_gdp_per_capita = gdp_per_capita_df[gdp_per_capita_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_inflation = inflation_df[inflation_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_unemployment = unemployment_df[unemployment_df["Country Name"].str.strip().str.lower() == target_country.lower()]

In [ ]:
vn_gdp_growth = vn_gdp_growth.dropna()
vn_gdp_per_capita  = vn_gdp_per_capita .dropna()
vn_df = vn_df.dropna()
vn_inflation = vn_inflation.dropna()
vn_unemployment = vn_unemployment.dropna()

In [ ]:
vn_df = vn_df.rename(columns={"Value": "GDP"})
vn_gdp_growth = vn_gdp_growth.rename(columns={"Value": "GDP_Growth"})
vn_gdp_per_capita = vn_gdp_per_capita.rename(columns={"Value": "GDP_Per_Capita"})
vn_inflation = vn_inflation.rename(columns={"Value": "Inflation"})
vn_unemployment = vn_unemployment.rename(columns={"Value": "Unemployment"})

In [ ]:
vn_merged = vn_df[["Year", "GDP"]].merge( # có từ 1985
    vn_gdp_growth[["Year", "GDP_Growth"]], # có từ 1985
    on="Year",
    how="inner"
).merge(
    vn_gdp_per_capita[["Year", "GDP_Per_Capita"]], # có từ 1985
    on="Year",
    how="inner"
).merge(
    vn_inflation[["Year", "Inflation"]], # có từ 1996
    on="Year",
    how="inner"
).merge(
    vn_unemployment[["Year", "Unemployment"]], # có từ 1991
    on="Year",
    how="inner"
)

vn_merged = vn_merged.sort_values("Year").reset_index(drop=True)

In [ ]:
vn_merged

In [115]:
vn_merged.to_csv("vietnam_macro.csv", index=False)